# 🐝 Hailo DFC Compilation - YOLO11m Bee Detection

This notebook compiles YOLO11m ONNX model to Hailo HEF format for Raspberry Pi + Hailo-8L

**Advantages over WSL:**
- ✅ GPU acceleration (T4)
- ✅ Higher optimization levels (2-3 vs 0)
- ✅ Better accuracy (~86-87% vs ~85-86%)
- ✅ Cloud-based, no local setup

**Runtime:** ~30-40 minutes total

---

## ⚙️ Setup Instructions:

1. **Enable GPU:** Runtime → Change runtime type → T4 GPU
2. **Upload files to Colab:**
   - `yolo11m_bee_best.onnx` (from Mac)
   - `hailo_dataflow_compiler-3.33.0-py3-none-linux_x86_64.whl` (from wsl_transfer folder)
3. **Run all cells**
4. **Download:** `yolo11m_bee.hef` at the end

## 📦 Step 1: Install Hailo Dataflow Compiler

In [1]:
# Check GPU availability
!nvidia-smi

Mon Oct 13 02:45:29 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# Upload Hailo DFC wheel file
from google.colab import files
import os

print("📤 Upload: hailo_dataflow_compiler-3.33.0-py3-none-linux_x86_64.whl")
print("   Location: /Users/davidgassier/digital4ai/bee-monitoring-system/wsl_transfer/")
print()

#uploaded = files.upload()

# Find the .whl file
#whl_file = [f for f in uploaded.keys() if f.endswith('.whl')][0]
whl_file = "hailo_dataflow_compiler-3.33.0-py3-none-linux_x86_64.whl"
print(f"\n✅ Uploaded: {whl_file}")

📤 Upload: hailo_dataflow_compiler-3.33.0-py3-none-linux_x86_64.whl
   Location: /Users/davidgassier/digital4ai/bee-monitoring-system/wsl_transfer/



NameError: name 'uploaded' is not defined

In [4]:
# Install Hailo DFC
print("📦 Installing Hailo Dataflow Compiler...")
print("   This may take 5-10 minutes...\n")

#!pip install -q {whl_file}

# Install system dependencies for pygraphviz
!apt-get update -qq
!apt-get install -y graphviz graphviz-dev pkg-config
!pip install pygraphviz --no-cache-dir

print("✅ System dependencies installed")

!pip install -q whl_file{whl_file}
print("\n✅ Hailo Dataflow Compiler installed")
#

# Verify installation
print("\n✅ Verifying installation...\n")
!hailo --version

📦 Installing Hailo Dataflow Compiler...
   This may take 5-10 minutes...

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
Note, selecting 'libgraphviz-dev' instead of 'graphviz-dev'
graphviz is already the newest version (2.42.2-6ubuntu0.1).
The following packages were automatically installed and are no longer required:
  libbz2-dev libpkgconf3 libreadline-dev
Use 'apt autoremove' to remove them.
The following additional packages will be installed:
  libgail-common libgail18 libgtk2.0-0 libgtk2.0-bin libgtk2.0-common
  libgvc6-plugins-gtk librsvg2-common libxdot4
Suggested packages:
  gvfs
The following packages will be REMOVED:
  pkgconf r-base-dev
The following NEW packages will be installed:
  libgail-common libgail18 libgraphviz-dev libgtk2.0-0 lib

## 📥 Step 2: Upload ONNX Model

In [6]:
# Upload YOLO11m ONNX model
from google.colab import files

print("📤 Upload: yolo11m_bee_best.onnx")
print("   Location: /Users/davidgassier/digital4ai/bee-monitoring-system/models/yolo11m/")
print("   Size: ~77 MB")
print()

#uploaded = files.upload()

# Find ONNX file
#onnx_file = [f for f in uploaded.keys() if f.endswith('.onnx')][0]

onnx_file = "yolo11m_bee_best.onnx"
print(f"\n✅ Uploaded: {onnx_file}")
print(f"   Size: {os.path.getsize(onnx_file) / 1024 / 1024:.1f} MB")

📤 Upload: yolo11m_bee_best.onnx
   Location: /Users/davidgassier/digital4ai/bee-monitoring-system/models/yolo11m/
   Size: ~77 MB


✅ Uploaded: yolo11m_bee_best.onnx
   Size: 76.7 MB


## 🔄 Step 3: Parse ONNX to HAR

In [7]:
# Parse ONNX to HAR
import subprocess
import sys

print("="*70)
print("🔄 Step 1/3: Parsing ONNX to HAR")
print("="*70)
print("\n⏱️  Expected time: 5-10 minutes\n")

# Note: This will prompt for end nodes - answer 'y' twice
# First prompt: Use recommended end node '/model.23/Concat_3' -> y
# Second prompt: Use native Hailo NMS post-processing -> y

#!hailo parser onnx {onnx_file} --hw-arch hailo8l
!hailo parser onnx "{onnx_file}" --hw-arch hailo8l

# Check HAR was created
har_file = onnx_file.replace('.onnx', '.har')
if os.path.exists(har_file):
    print(f"\n✅ HAR created: {har_file}")
    print(f"   Size: {os.path.getsize(har_file) / 1024 / 1024:.1f} MB")
else:
    print("\n❌ HAR file not found! Check output above for errors.")

🔄 Step 1/3: Parsing ONNX to HAR

⏱️  Expected time: 5-10 minutes

[info] No GPU chosen, Selected GPU 0
[info] Current Time: 03:01:37, 10/13/25
[info] CPU: Architecture: x86_64, Model: Intel(R) Xeon(R) CPU @ 2.20GHz, Number Of Cores: 2, Utilization: 1.5%
[info] Memory: Total: 12GB, Available: 10GB
[info] System info: OS: Linux, Kernel: 6.6.97+
[info] Hailo DFC Version: 3.33.0
[info] HailoRT Version: Not Installed
[info] PCIe: No Hailo PCIe device was found
[info] Running `hailo parser onnx yolo11m_bee_best.onnx --hw-arch hailo8l`
[info] Translation started on ONNX model yolo11m_bee_best
[info] Restored ONNX model yolo11m_bee_best (completion time: 00:00:00.94)
[info] Extracted ONNXRuntime meta-data for Hailo model (completion time: 00:00:02.79)
[info] Simplified ONNX model for a parsing retry attempt (completion time: 00:00:06.01)
Parsing failed with recommendations for end node names: ['/model.23/Concat_3'].
Would you like to parse again with the recommendation? (y/n) 
y
[info] Accordi

## ⚡ Step 4: Optimize HAR (Quantization)

**With GPU:** This will use optimization level 2-3 (much better than WSL's level 0!)

In [ ]:
# Optimize with random calibration
print("="*70)
print("⚡ Step 2/3: Optimizing HAR (Quantization)")
print("="*70)
print("\n⏱️  Expected time: 15-20 minutes with GPU\n")

optimized_har = har_file.replace('.har', '_optimized.har')

!hailo optimize {har_file} \
    --hw-arch hailo8l \
    --use-random-calib-set \
    --output-har-path {optimized_har}

# Check optimized HAR
if os.path.exists(optimized_har):
    print(f"\n✅ Optimized HAR created: {optimized_har}")
    print(f"   Size: {os.path.getsize(optimized_har) / 1024 / 1024:.1f} MB")
else:
    print("\n❌ Optimized HAR not found! Check output above.")

⚡ Step 2/3: Optimizing HAR (Quantization)

⏱️  Expected time: 15-20 minutes with GPU

[info] No GPU chosen, Selected GPU 0
[info] Current Time: 03:02:42, 10/13/25
[info] CPU: Architecture: x86_64, Model: Intel(R) Xeon(R) CPU @ 2.20GHz, Number Of Cores: 2, Utilization: 3.5%
[info] Memory: Total: 12GB, Available: 10GB
[info] System info: OS: Linux, Kernel: 6.6.97+
[info] Hailo DFC Version: 3.33.0
[info] HailoRT Version: Not Installed
[info] PCIe: No Hailo PCIe device was found
[info] Running `hailo optimize yolo11m_bee_best.har --hw-arch hailo8l --use-random-calib-set --output-har-path yolo11m_bee_best_optimized.har`
[info] For NMS architecture yolov8 the default engine is cpu. For other engine please use the 'engine' flag in the nms_postprocess model script command. If the NMS has been added during parsing, please parse the model again without confirming the addition of the NMS, and add the command manually with the desired engine.
[info] The layer yolo11m_bee_best/conv74 was detected a

## 🚀 Step 5: Compile HAR to HEF

In [ ]:
# Compile to HEF
print("="*70)
print("🚀 Step 3/3: Compiling HAR to HEF")
print("="*70)
print("\n⏱️  Expected time: 10-15 minutes\n")

!hailo compiler {optimized_har} --hw-arch hailo8l

# Find generated HEF
hef_file = optimized_har.replace('.har', '.hef')
final_hef = 'yolo11m_bee.hef'

if os.path.exists(hef_file):
    # Rename to simpler name
    import shutil
    shutil.move(hef_file, final_hef)
    print(f"\n✅ HEF created: {final_hef}")
    print(f"   Size: {os.path.getsize(final_hef) / 1024 / 1024:.1f} MB")
else:
    print("\n❌ HEF file not found! Check output above.")

## 📊 Step 6: Verify HEF Model

In [ ]:
# Parse and display HEF info
print("="*70)
print("📊 HEF Model Information")
print("="*70)
print()

!hailo parser {final_hef}

print("\n" + "="*70)
print("✅ COMPILATION COMPLETE!")
print("="*70)
print()
print(f"📦 Output: {final_hef}")
print(f"📏 Size: {os.path.getsize(final_hef) / 1024 / 1024:.1f} MB")
print()
print("🎯 Next Steps:")
print("   1. Download the HEF file below")
print("   2. Transfer to Raspberry Pi:")
print("      scp yolo11m_bee.hef digital4ai@192.168.68.66:/opt/bee-monitoring/models/")
print("   3. Test inference on Pi")
print()
print("⚡ Expected Performance on Hailo-8L:")
print("   - Inference: 25-33ms per frame")
print("   - FPS: 30-40")
print("   - Accuracy: ~86-87% mAP50 (with GPU optimization!)")
print()

## 📥 Step 7: Download HEF File

In [ ]:
# Download HEF file
from google.colab import files

print("📥 Downloading HEF file...\n")

if os.path.exists(final_hef):
    files.download(final_hef)
    print(f"\n✅ Download started: {final_hef}")
    print("\n🎯 Transfer to Raspberry Pi:")
    print(f"   scp {final_hef} digital4ai@192.168.68.66:/opt/bee-monitoring/models/")
else:
    print(f"❌ File not found: {final_hef}")

## 🔄 Optional: Compile with Real Calibration Images

For +1-2% better accuracy, upload 50-100 bee images and re-optimize:

In [ ]:
# Upload calibration images (optional)
# Uncomment and run if you want better accuracy

# # Create calibration directory
# !mkdir -p calibration_images
#
# print("📤 Upload 50-100 bee images for calibration")
# print("   Upload to: calibration_images/\n")
#
# # Upload images
# uploaded = files.upload()
#
# # Move to calibration dir
# for filename in uploaded.keys():
#     !mv {filename} calibration_images/
#
# # Re-optimize with real images
# !hailo optimize {har_file} \
#     --hw-arch hailo8l \
#     --calib-set-path calibration_images \
#     --output-har-path yolo11m_bee_calibrated.har
#
# # Re-compile
# !hailo compiler yolo11m_bee_calibrated.har --hw-arch hailo8l
#
# print("\n✅ Calibrated HEF created: yolo11m_bee_calibrated.hef")

---

## 📝 Notes

**Advantages of Colab Compilation:**
- ✅ GPU acceleration (better optimization)
- ✅ Higher accuracy than WSL CPU-only
- ✅ Cloud-based, reproducible
- ✅ Free to use

**Considerations:**
- ⏱️ Session timeout: 12 hours (enough for compilation)
- 📤 Need to upload/download files
- 💾 Files deleted when session ends (download HEF!)

**Performance Comparison:**
- WSL (CPU, opt level 0): ~85-86% mAP, 30-40 FPS
- Colab (GPU, opt level 2-3): ~86-87% mAP, 30-40 FPS

---

**Created by:** David Gassier  
**Project:** Bee Monitoring System  
**Model:** YOLO11m trained on BeeMasterV2 dataset  
**Target:** Raspberry Pi 5 + Hailo-8L (13 TOPS)
